# GR Calculator - Google Colab Edition

**Symbolic General Relativity Analysis powered by SymPy**

### Quick-start
1. **Cell 1** - install dependencies (run once per session)
2. **Cell 2** - clone the GitHub repo and import the modules
3. **Cell 3** - choose the document metric variant, profile, and report options
4. **Cell 4** - run all symbolic GR computations
5. **Cell 5** - generate the run report, then optionally export a separate comparison report
6. **Cell 6** - numerical evaluation of scalars on a grid (only for explicit profiles)
7. **Cell 7** - visualise GR scalars as 2-D heat maps
8. **Cell 8** - geodesic integration and trajectory plot

> **Report note**: `gr_report.pdf` is the direct output of the symbolic run. If enabled,
> `gr_comparison_report.pdf` is generated separately for checks against external formulas.

> **GPU note**: SymPy symbolic computation always runs on CPU. GPU (CuPy / JAX)
> accelerates *numerical* post-processing in cells 6-8.
> To enable, select *Runtime -> Change runtime type -> GPU* before starting.



In [ ]:
# ============================================================
# Cell 1 - Install dependencies (run once per Colab session)
# ============================================================

# LaTeX for PDF generation
!apt-get update -qq
!apt-get install -y texlive-latex-base texlive-latex-extra texlive-fonts-recommended texlive-science lmodern dvipng > /dev/null 2>&1
!which pdflatex

# Core Python packages
!pip install -q sympy matplotlib scipy

# --- Optional GPU packages ---------------------------------------------------
# Uncomment ONE of the blocks below if you selected a GPU runtime:

# CuPy (NVIDIA CUDA - fastest for array operations):
# !pip install -q cupy-cuda12x

# JAX + diffrax (for JIT-compiled geodesic integration):
# !pip install -q "jax[cuda12]" diffrax
# ----------------------------------------------------------------------------

print('Setup complete.')


In [ ]:
# ============================================================
# Cell 2 - Clone repo and import modules
# ============================================================
import os
import shutil
import subprocess
import sys

REPO_URL = 'https://github.com/xys004/GR_python.git'
REPO_BRANCH = 'main'   # change this only if you want to test another branch
REPO_DIR = '/content/GR_python'
MODULE_DIR = os.path.join(REPO_DIR, 'GR_python_colab')
ROOT_MODULE_DIR = REPO_DIR

if os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)

subprocess.run(
    ['git', 'clone', '--depth', '1', '--branch', REPO_BRANCH, REPO_URL, REPO_DIR],
    check=True,
)

for path in (MODULE_DIR, ROOT_MODULE_DIR):
    if path not in sys.path:
        sys.path.insert(0, path)

from gr_tensors import progress
from gr_latex import assemble_report, assemble_comparison_report
import gr_main as grm
import gr_numerics as grn
import gr_warp as grw

print('Colab modules loaded from:', MODULE_DIR)
print('Shared modules loaded from:', ROOT_MODULE_DIR)
print('Repo branch:', REPO_BRANCH)
print('GPU backend detected:', grn.detect_backend())


In [ ]:
# ============================================================
# Cell 3 - User input  *** EDIT THIS CELL ***
# ============================================================
import sympy as sp
from sympy import Function, symbols, tanh

# --- Coordinate symbols ------------------------------------------------------
t, r, theta, phi = symbols('t r theta phi', real=True)
coords = [t, r, theta, phi]
dim    = 4

# --- Variant selection -------------------------------------------------------
# Options: 'baseline', 'variant_a', 'variant_b'
# Recommended presets:
#   first symbolic/document check -> VARIANT='variant_a', PROFILE_MODE='document_generic'
#   first numerical demo          -> VARIANT='baseline', PROFILE_MODE='schwarzschild_pg'
VARIANT = 'variant_a'

# Options:
#   'document_generic' -> beta(r) and B(r) stay symbolic for direct formula checks
#   'schwarzschild_pg' -> beta = sqrt(2M/r), with optional explicit B(r) for plots
PROFILE_MODE = 'document_generic'

# --- Reporting switches ------------------------------------------------------
# Main report: always the direct symbolic output of the current run.
# Comparison report: optional, kept separate so provenance stays clear.
RUN_DOCUMENT_COMPARISON = True
GENERATE_COMPARISON_REPORT = True

# --- Parameters --------------------------------------------------------------
M = symbols('M', positive=True)
M_num = 1.0
B_expr = None

if PROFILE_MODE == 'document_generic':
    beta_expr = Function('beta')(r)
    if VARIANT != 'baseline':
        B_expr = Function('B')(r)
    parameter_values = {}
else:
    beta_expr = sp.sqrt(2 * M / r)
    if VARIANT == 'variant_a':
        B_expr = 1 + 4 * (1 - tanh((r - 4) / sp.Rational(4, 5))) / 2
    elif VARIANT == 'variant_b':
        B_expr = 1 + 4 * (1 - tanh((r - 4) / sp.Rational(4, 5))) / 2
    parameter_values = {M: M_num}

metric_config = grw.build_metric_configuration(VARIANT, coords, beta_expr, B_expr)

grm.g_metric            = metric_config['g_metric']
grm.coords              = coords
grm.dim                 = dim
grm.g_inv_metric        = None
grm.e_tetrad            = metric_config['e_tetrad']
grm.latex_subs          = {}
grm.METRIC_NAME         = metric_config['metric_name']
grm.METRIC_DESCRIPTION  = metric_config['description']
grm.AUTHOR_NAME         = 'Colab GR Engine'
grm.OUTPUT_FILENAME     = 'gr_report'
grm.COMPARISON_OUTPUT_FILENAME = 'gr_comparison_report'

grm.DOCUMENT_VARIANT          = metric_config['variant']
grm.DOCUMENT_PROFILE_MODE     = PROFILE_MODE
grm.DOCUMENT_beta_expr        = beta_expr
grm.DOCUMENT_B_expr           = B_expr
grm.DOCUMENT_parameter_values = parameter_values
grm.RUN_DOCUMENT_COMPARISON   = RUN_DOCUMENT_COMPARISON
grm.GENERATE_COMPARISON_REPORT = GENERATE_COMPARISON_REPORT

grm.COMPUTE_WEYL        = True
grm.COMPUTE_KRETSCHMANN = True
grm.COMPUTE_GEODESICS   = True
grm.COMPUTE_KILLING     = True
grm.COMPUTE_TETRAD      = True
grm.FAST_MODE           = False

grm.USE_PARALLEL    = False
grm.N_PARALLEL_JOBS = 2
grm.USE_GPU         = False

print('Configuration set.')
print('Variant:', grm.DOCUMENT_VARIANT)
print('Profile mode:', PROFILE_MODE)
print('Metric:', grm.METRIC_NAME)
print('beta(r) =', beta_expr)
print('B(r)    =', B_expr if B_expr is not None else 1)
print('Numerical substitutions:', parameter_values)
print('Run comparison:', RUN_DOCUMENT_COMPARISON)
print('Generate comparison report:', GENERATE_COMPARISON_REPORT)
if PROFILE_MODE == 'document_generic':
    print("Tip: use PROFILE_MODE = 'schwarzschild_pg' later if you want numerical plots.")



In [ ]:
# ============================================================
# Cell 4 — Run symbolic GR computations
# ============================================================
# This is the main computation step and may take several minutes
# for complex metrics.  Progress messages are printed to the console.

results = grm.run_computations(
    g_metric                 = grm.g_metric,
    coords                   = grm.coords,
    dim                      = grm.dim,
    g_inv_metric             = grm.g_inv_metric,
    e_tetrad                 = grm.e_tetrad,
    compute_weyl_flag        = grm.COMPUTE_WEYL,
    compute_kretschmann_flag = grm.COMPUTE_KRETSCHMANN,
    compute_geodesics_flag   = grm.COMPUTE_GEODESICS,
    compute_killing_flag     = grm.COMPUTE_KILLING,
    compute_tetrad_flag      = grm.COMPUTE_TETRAD,
    fast_mode                = grm.FAST_MODE,
    compute_parallel         = grm.USE_PARALLEL,
    n_jobs                   = grm.N_PARALLEL_JOBS,
)

print('\nSymbolic computation complete.')
print('Keys in results:', list(results.keys()))

In [ ]:
# ============================================================
# Cell 5 - Generate the run report and, optionally, a separate comparison report
# ============================================================
# The LaTeX sources and PDFs are saved to /content/.

from IPython.display import FileLink, display

def show_artifact(label, base_path):
    tex_path = base_path + '.tex'
    pdf_path = base_path + '.pdf'

    if os.path.exists(tex_path):
        print(f'{label} LaTeX generated: {tex_path}')
        display(FileLink(tex_path))
    else:
        print(f'{label} LaTeX source not found.')

    if os.path.exists(pdf_path):
        print(f'{label} PDF generated: {pdf_path}')
        display(FileLink(pdf_path))
    else:
        print(f'{label} PDF not found - check the .log file for LaTeX errors.')

# --- Main run report ---------------------------------------------------------
report_lines = assemble_report(
    results         = results,
    coords          = grm.coords,
    dim             = grm.dim,
    metric_name     = grm.METRIC_NAME,
    description     = grm.METRIC_DESCRIPTION,
    author          = grm.AUTHOR_NAME,
    latex_subs_dict = grm.latex_subs,
    g_metric_orig   = grm.g_metric,
    e_tetrad        = grm.e_tetrad,
)

run_output_path = os.path.join('/content/', grm.OUTPUT_FILENAME)
grm.write_and_compile_pdf(report_lines, run_output_path)
show_artifact('Run report', run_output_path)

# --- Optional document comparison -------------------------------------------
comparison = None
if grm.RUN_DOCUMENT_COMPARISON:
    comparison = grw.compare_document_formulas(
        results,
        grm.DOCUMENT_VARIANT,
        r,
        grm.DOCUMENT_beta_expr,
        grm.DOCUMENT_B_expr,
    )

    grw.print_formula_comparison(comparison)
    for key in ('rho', 'p_r', 'p_perp', 'q'):
        if not comparison['checks'][key]:
            print(f"Residual for {key}:", comparison['residuals'][key])

    if grm.GENERATE_COMPARISON_REPORT:
        comparison_lines = assemble_comparison_report(
            comparison      = comparison,
            coords          = grm.coords,
            metric_name     = grm.METRIC_NAME,
            description     = grm.METRIC_DESCRIPTION,
            author          = grm.AUTHOR_NAME,
            latex_subs_dict = grm.latex_subs,
            beta_expr       = grm.DOCUMENT_beta_expr,
            B_expr          = grm.DOCUMENT_B_expr,
            profile_mode    = grm.DOCUMENT_PROFILE_MODE,
        )
        comparison_output_path = os.path.join('/content/', grm.COMPARISON_OUTPUT_FILENAME)
        grm.write_and_compile_pdf(comparison_lines, comparison_output_path)
        show_artifact('Comparison report', comparison_output_path)
    else:
        print('Comparison report generation is disabled in Cell 3.')
else:
    print('Document comparison is disabled in Cell 3.')



In [ ]:
# ============================================================
# Cell 6 - Numerical evaluation of GR scalars on a grid
# ============================================================
# Numerical plots require an explicit profile (not abstract beta(r), B(r)).

import numpy as np

if grm.DOCUMENT_PROFILE_MODE == 'document_generic':
    num_results = {}
    print('Numerical evaluation skipped: PROFILE_MODE=document_generic uses abstract functions.')
    print("Switch Cell 3 to PROFILE_MODE = 'schwarzschild_pg' or define explicit beta/B profiles.")
else:
    grid_npts = 40
    coord_ranges = {
        t:     (0.0, 10.0),
        r:     (0.5, 20.0),
        theta: (0.1, 3.0),
        phi:   (0.0, 6.283),
    }

    parameter_values = grm.DOCUMENT_parameter_values

    num_results = grn.evaluate_results_numerical(
        results, grm.coords, coord_ranges, npts=grid_npts, parameter_subs=parameter_values
    )

    if 'K' not in num_results and 'Schwarzschild' in grm.METRIC_NAME:
        print('K could not be evaluated from the symbolic result; using Schwarzschild fallback K = 48 M^2 / r^6.')
        coord_1d = [
            np.linspace(*coord_ranges.get(c, (0.0, 1.0)), grid_npts)
            for c in grm.coords
        ]
        grids = np.meshgrid(*coord_1d, indexing='ij')
        rr = grids[1]
        num_results['K'] = 48.0 * (float(parameter_values[M]) ** 2) / (rr ** 6)

    print('Numerically evaluated scalars:', list(num_results.keys()))
    for key, arr in num_results.items():
        a = np.array(arr)
        print(f'  {key}: shape={a.shape}, min={float(a.min()):.4g}, max={float(a.max()):.4g}')


In [ ]:
# ============================================================
# Cell 7 - Visualise GR scalars as 2-D heat maps
# ============================================================
# Try r-theta first, then fall back automatically if needed.

import numpy as np

preferred_axes = (1, 2)
fallback_axes = (0, 1)


def choose_axes(coords, preferred=(1, 2), fallback=(0, 1)):
    ncoords = len(coords)
    valid_pairs = [(i, j) for i in range(ncoords) for j in range(i + 1, ncoords)]
    if preferred in valid_pairs:
        return preferred
    if fallback in valid_pairs:
        return fallback
    return valid_pairs[0]


filtered_results = {}
for key, arr in num_results.items():
    arr_np = np.array(arr)
    if arr_np.size == 0:
        continue
    if np.allclose(arr_np, arr_np.flat[0]):
        print(f"Skipping {key}: scalar is constant on the current grid.")
        continue
    filtered_results[key] = arr

if not filtered_results:
    print("No varying scalars are available to plot on the current grid.")
else:
    axes_to_use = choose_axes(grm.coords, preferred_axes, fallback_axes)
    axis_names = [str(grm.coords[i]) for i in axes_to_use]
    print("Using slice_axes =", axes_to_use, "->", axis_names)
    grn.plot_gr_quantities(filtered_results, grm.coords, slice_axes=axes_to_use)


In [ ]:
# ============================================================
# Cell 8 - Geodesic integration template (optional / advanced)
# ============================================================
# This cell demonstrates the geodesic integration API with a simple
# placeholder RHS. It is a template, not a physically correct geodesic
# unless you replace the accelerations with lambdified Christoffel terms
# from results['Gamma'] or results['geodesics'].
#
# State vector: y = [x0, x1, x2, x3, v0, v1, v2, v3]
#               where xi = coord position, vi = dx^i/dtau

import numpy as np

# Example: Schwarzschild equatorial circular orbit at r=6M (M=1 numerically)
# For a proper geodesic replace the placeholder rhs below with lambdified
# Christoffel symbols from results['Gamma'].
r0    = 6.0                     # orbital radius (in units of M)
M_val = 1.0
f0    = 1.0 - 2.0 * M_val / r0
Omega = np.sqrt(M_val / r0**3)  # angular velocity of circular orbit

y0 = [
    0.0,            # t(0)
    r0,             # r(0)
    np.pi / 2,      # theta(0) = equatorial plane
    0.0,            # phi(0)
    1.0 / f0,       # dt/dtau  (normalised so g_{mu nu} u^mu u^nu = -1)
    0.0,            # dr/dtau  (circular orbit)
    0.0,            # dtheta/dtau
    Omega / f0,     # dphi/dtau
]

def rhs(tau, y):
    """
    Placeholder RHS - replaces d^2 x^i/dtau^2 = -Gamma^i_{jk} u^j u^k
    with zero acceleration (free-fall placeholder).

    *** Replace the zeros with lambdified Christoffel terms for a real geodesic. ***
    """
    positions  = y[:4]
    velocities = y[4:]
    # dy/dtau = [v0, v1, v2, v3, a0, a1, a2, a3]
    accelerations = [0.0, 0.0, 0.0, 0.0]   # <-- replace with -Gamma terms
    return list(velocities) + accelerations

tau_arr, y_arr = grn.integrate_geodesic_jax(
    rhs_fn   = rhs,
    y0       = y0,
    tau_span = (0.0, 2.0 * np.pi / Omega),   # one full orbit
    n_steps  = 500,
)

print('Template integration complete.')
print('This is not a physical geodesic unless rhs() was replaced by the true geodesic RHS.')
print(f'tau range: {tau_arr[0]:.2f} -> {tau_arr[-1]:.2f}')
grn.plot_geodesic(tau_arr, y_arr, grm.coords)
